# Qwen2.5-7B 训练全流程

这个 notebook 集中说明 Qwen2.5-7B 的 SFT、PEFT、偏好优化和奖励优化。

对应方法：

- SFT：监督微调。
- LoRA：PEFT 的常用方式。
- QLoRA：4bit 加载基座模型并训练 LoRA adapter。
- DPO：用 chosen/rejected 偏好数据优化模型。
- PPO：用奖励模型或奖励函数优化模型。
- GRPO：用组内相对奖励优化模型，适合可验证任务。

这里的代码是真实代码骨架，但训练大模型会下载权重并占用大量显存。需要执行时，先确认 `00_env_check.ipynb` 通过。

## 1. 基础配置

这部分定义所有训练方法都会用到的路径和参数。

- `model_name`：基座模型名称。
- `sft_data_path`：SFT 数据。
- `dpo_data_path`：DPO 偏好数据。
- `rl_data_path`：PPO/GRPO 奖励数据。
- `output_root`：训练产物输出目录。
- `max_seq_length`：训练样本最大 token 长度，越大越吃显存。

In [ ]:
from pathlib import Path

model_name = "Qwen/Qwen2.5-7B-Instruct"
sft_data_path = "../../data/sample_sft.jsonl"
dpo_data_path = "../../data/sample_preference_dpo.jsonl"
rl_data_path = "../../data/sample_prompts_rl.jsonl"
output_root = Path("../../outputs/qwen2_5_7b")
max_seq_length = 2048

output_root.mkdir(parents=True, exist_ok=True)

## 2. 读取数据

本仓库的样例数据是 JSONL。每一行都是一个完整 JSON 对象，可以直接用 `datasets` 读取。

SFT、LoRA、QLoRA 使用 `messages` 格式。DPO 使用 `prompt/chosen/rejected` 格式。PPO/GRPO 使用 `prompt/answer/reward_type` 格式。

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files=sft_data_path, split="train")
dpo_dataset = load_dataset("json", data_files=dpo_data_path, split="train")
rl_dataset = load_dataset("json", data_files=rl_data_path, split="train")

print(sft_dataset[0])
print(dpo_dataset[0])
print(rl_dataset[0])

## 3. SFT 是什么

SFT 是监督微调。它的目标是让模型看到输入后，学习生成指定的 assistant 输出。

适合场景：

- 想让模型学习固定回答格式。
- 想让模型适应某个领域的话术。
- 想让模型更稳定地遵守指令。

不适合场景：

- 只是想让模型知道大量新事实。大量事实更适合检索增强或专门的数据策略。
- 没有高质量目标回答。SFT 会直接学习你的答案质量。

### 3.1 SFT 关键参数

- `per_device_train_batch_size`：单张 GPU 每步处理多少样本。显存不足时先调小。
- `gradient_accumulation_steps`：梯度累积步数。batch 小时用它模拟更大 batch。
- `learning_rate`：学习率。太大容易训坏，太小收敛慢。
- `num_train_epochs`：遍历数据集的轮数。小样例只用于流程验证，真实数据要看验证集表现。
- `max_seq_length`：最大 token 长度。调大能看更长上下文，但显存占用明显增加。
- `bf16`：是否使用 bfloat16。支持 bf16 的 GPU 通常优先选它。
- `gradient_checkpointing`：用计算换显存。省显存但训练更慢。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

sft_args = TrainingArguments(
    output_dir=str(output_root / "sft"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

# 需要真实训练时取消下一行注释。
# sft_trainer.train()

## 4. LoRA 是什么

LoRA 属于 PEFT。它冻结基座模型参数，只在指定线性层旁边训练低秩矩阵。

它解决的问题是：全量微调显存和存储成本太高。LoRA adapter 通常比完整模型小很多，便于保存、切换和分享。

常见误区：LoRA adapter 不是完整模型。部署时要么和基座模型一起加载，要么先 merge 到基座模型。

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

lora_args = TrainingArguments(
    output_dir=str(output_root / "lora"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
)

lora_trainer = SFTTrainer(
    model=model,
    args=lora_args,
    train_dataset=sft_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# 需要真实训练时取消下一行注释。
# lora_trainer.train()

## 5. QLoRA 是什么

QLoRA 是“4bit 加载基座模型 + 训练 LoRA adapter”。它不是把 adapter 训练成 4bit，也不是直接输出一个 4bit 完整模型。

适合场景：显存有限，但仍希望训练 7B 或更大模型。

代价：4bit 加载会引入额外处理，训练速度可能慢一些；bitsandbytes 和 CUDA 环境必须正常。

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

qlora_args = TrainingArguments(
    output_dir=str(output_root / "qlora"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    args=qlora_args,
    train_dataset=sft_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# 需要真实训练时取消下一行注释。
# qlora_trainer.train()

## 6. DPO 是什么

DPO 用偏好数据训练模型。每条数据包含同一个问题下的好回答 `chosen` 和差回答 `rejected`。

它适合已经有人类偏好、规则偏好或模型打分偏好数据的场景。它不需要先训练奖励模型，因此比 PPO 更容易开始。

关键参数 `beta` 控制偏好约束强度。太小偏好学习弱，太大可能让模型变化过猛。

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir=str(output_root / "dpo"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-7,
    num_train_epochs=1,
    logging_steps=1,
    beta=0.1,
    max_prompt_length=1024,
    max_completion_length=1024,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

# 需要真实训练时取消下一行注释。
# dpo_trainer.train()

## 7. PPO 是什么

PPO 是经典 RLHF 路线。它通常需要策略模型、参考模型、奖励模型或奖励函数，并用 KL 惩罚限制模型不要偏离原模型太远。

PPO 不是新人第一步。它重要，但组件多、训练稳定性要求更高。本仓库会先用规则奖励函数解释流程，再扩展到奖励模型。

In [ ]:
import json
import re

def rule_reward(response: str, answer: str, reward_type: str) -> float:
    text = response.strip()
    if reward_type == "exact_match":
        return 1.0 if text == answer else 0.0
    if reward_type == "regex_match":
        pattern = answer.removeprefix("regex:")
        return 1.0 if re.fullmatch(pattern, text) else 0.0
    if reward_type == "json_schema":
        try:
            return 1.0 if json.loads(text) == json.loads(answer) else 0.0
        except json.JSONDecodeError:
            return 0.0
    raise ValueError(f"unknown reward_type: {reward_type}")

rule_reward("42", rl_dataset[0]["answer"], rl_dataset[0]["reward_type"])

## 8. GRPO 是什么

GRPO 会对同一个 prompt 生成多个候选回答，然后根据组内相对奖励更新模型。它适合数学、代码、JSON 格式、正则匹配这类可验证任务。

和 PPO 相比，GRPO 的一个重要吸引力是减少 value model 相关成本。但它仍然要求奖励函数可靠，否则模型会优化错误目标。

关键参数：

- `num_generations`：每个 prompt 生成几个候选。越大越能比较，但显存和时间成本越高。
- `temperature`：生成候选时的随机性。太低候选过于相似，太高可能质量不稳定。
- `max_completion_length`：每个候选回答的最大长度。

In [ ]:
from trl import GRPOConfig, GRPOTrainer

def grpo_reward_func(prompts, completions, answer=None, reward_type=None, **kwargs):
    scores = []
    answers = answer or [None] * len(completions)
    reward_types = reward_type or ["exact_match"] * len(completions)
    for completion, expected, kind in zip(completions, answers, reward_types):
        if isinstance(completion, list):
            text = completion[-1].get("content", "")
        else:
            text = str(completion)
        scores.append(rule_reward(text, expected, kind))
    return scores

grpo_config = GRPOConfig(
    output_dir=str(output_root / "grpo"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-6,
    num_train_epochs=1,
    logging_steps=1,
    num_generations=2,
    max_completion_length=256,
    temperature=0.7,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=rl_dataset,
    reward_funcs=grpo_reward_func,
    processing_class=tokenizer,
)

# 需要真实训练时取消下一行注释。
# grpo_trainer.train()

## 9. 产物如何衔接

- SFT checkpoint 可以继续做 DPO，也可以直接评估。
- LoRA/QLoRA 产物是 adapter，部署时需要基座模型配合，或先 merge。
- DPO 后的 adapter 可以继续评估，也可以再做 GRPO。
- GRPO/PPO 后的模型要重点检查是否过度优化奖励函数。
- 准备部署前，先用 Transformers 验证，再考虑 vLLM、SGLang、Ollama、llama.cpp。